<a href="https://colab.research.google.com/github/342olive/scraped-data/blob/main/NLP-Applications/07_pipeline_steps_breakdown.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is probably the code that I copy-pasted from the book and then re factored subsequently


In [1]:

import pandas as pd
from datasets import load_dataset
from sklearn.model_selection import StratifiedKFold

def create_mini_imdb():
    print("Downloading IMDB dataset (this might take a moment)...")
    # Load the IMDB dataset from Hugging Face
    dataset = load_dataset("imdb")

     # Convert to pandas dataframe
    # We take only the train split for simplicity
    df = pd.DataFrame(dataset['train'])

     # --- CREATE SMALL SUBSET ---
    # We select 1000 positive and 1000 negative reviews for a total of 2000
    pos_df = df[df['label'] == 1].sample(1000, random_state=42)
    neg_df = df[df['label'] == 0].sample(1000, random_state=42)

    # Combine and shuffle
    df_small = pd.concat([pos_df, neg_df]).sample(frac=1, random_state=42).reset_index(drop=True)

    # Rename columns to match your train.py requirements
    # IMDB dataset has 'text' and 'label', your code expects 'review' and 'sentiment'
    df_small = df_small.rename(columns={'text': 'review', 'label': 'sentiment'})

    print(f"Created mini dataset with {len(df_small)} reviews.")

    # --- CREATE FOLDS ---
    print("Creating folds...")
    df_small["kfold"] = -1

    # Initialize StratifiedKFold
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Populate the kfold column
    for fold, (train_idx, val_idx) in enumerate(kf.split(X=df_small, y=df_small.sentiment.values)):
        df_small.loc[val_idx, 'kfold'] = fold

    # Save to CSV
    output_path = "../input/imdb_folds.csv"
    # Ensure directory exists (optional based on your setup)
    import os
    os.makedirs("../input", exist_ok=True)

    df_small.to_csv(output_path, index=False)
    print(f"Success! File saved to: {output_path}")
    print(df_small.head())

if __name__ == "__main__":
    create_mini_imdb()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Created mini dataset with 2000 reviews.
Creating folds...
Success! File saved to: ../input/imdb_folds.csv
                                              review  sentiment  kfold
0  The Good: I liked this movie because it was th...          0      2
1  I enjoy movies like this for their spirit, no ...          1      0
2  Slaughter High the tale of revenge by a nerdy ...          0      3
3  I have a six month old baby at home and time t...          1      3
4  When will the hurting stop? I never want to se...          0      3


In [2]:
# And finally, we have config.py
# config.py
# we define all the configuration here
MAX_LEN = 128
TRAIN_BATCH_SIZE = 16
VALID_BATCH_SIZE = 8
EPOCHS = 10

In [3]:

# engine.py
import torch
import torch.nn as nn

def train(data_loader, model, optimizer, device):
    """
    Train the model for one epoch.
    """
    model.train()

    for data in data_loader:
        reviews = data["review"].to(device, dtype=torch.long)
        targets = data["target"].to(device, dtype=torch.float)

        optimizer.zero_grad()

        predictions = model(reviews)

        loss = nn.BCEWithLogitsLoss()(
            predictions,
            targets.view(-1, 1)
        )

        loss.backward()
        optimizer.step()


def evaluate(data_loader, model, device):
    """
    Evaluate the model: returns predictions and targets.
    """
    final_predictions = []
    final_targets = []

    model.eval()

    with torch.no_grad():
        for data in data_loader:
            reviews = data["review"].to(device, dtype=torch.long)
            targets = data["target"].to(device, dtype=torch.float)

            predictions = model(reviews)

            predictions = predictions.cpu().numpy().tolist()
            targets = targets.cpu().numpy().tolist()

            final_predictions.extend(predictions)
            final_targets.extend(targets)

    return final_predictions, final_targets
#train() and evaluate() are called inside the train.py file

In [4]:
# lstm.py
import torch
import torch.nn as nn

class LSTM(nn.Module):
    def __init__(self, embedding_matrix):
        super(LSTM, self).__init__()

        num_words = embedding_matrix.shape[0]
        embed_dim = embedding_matrix.shape[1]

        self.embedding = nn.Embedding(
            num_embeddings=num_words,
            embedding_dim=embed_dim
        )

        self.embedding.weight = nn.Parameter(
            torch.tensor(
                embedding_matrix,
                dtype=torch.float32
            )
        )
        self.embedding.weight.requires_grad = False

        self.lstm = nn.LSTM(
            embed_dim,
            128,
            bidirectional=True,
            batch_first=True,
        )

        self.out = nn.Linear(512, 1)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm(x)

        avg_pool = torch.mean(x, 1)
        max_pool, _ = torch.max(x, 1)

        out = torch.cat((avg_pool, max_pool), 1)
        out = self.out(out)
        return out


In [5]:
# dataset.py
import torch
class IMDBDataset:
def __init__(self, reviews, targets):
"""
:param reviews: this is a numpy array
:param targets: a vector, numpy array
"""
self.reviews = reviews
self.targets = targets  # I changed self.target to self.targets
def __len__(self):
# returns length of the dataset
return len(self.reviews)
def __getitem__(self, item):
# for any given item, which is an int,
# return review and targets as torch tensor
# item is the index of the item in concern
review = self.reviews[item, :]
target = self.target[item]
return {
    "review": torch.tensor(review, dtype=torch.long),
"target": torch.tensor(target, dtype=torch.float)
}

IndentationError: expected an indented block after class definition on line 3 (2758091503.py, line 4)

In [ ]:
# train.py
# These functions will help us in train.py which is used for training multiple folds


# train.py
import io
import torch
import numpy as np
import pandas as pd
# yes, we use tensorflow
# but not for training the model!
import tensorflow as tf
from sklearn import metrics
import config
import dataset
import engine
import lstm
def load_vectors(fname):
# taken from: https://fasttext.cc/docs/en/english-vectors.html
fin = io.open(
fname,
'r',
encoding='utf-8',
newline='\n',
errors='ignore'
)
n, d = map(int, fin.readline().split())
data = {}
for line in fin:
tokens = line.rstrip().split(' ')
data[tokens[0]] = list(map(float, tokens[1:]))
return data
def create_embedding_matrix(word_index, embedding_dict):
"""
This function creates the embedding matrix.
:param word_index: a dictionary with word:index_value
:param embedding_dict: a dictionary with word:embedding_vector
:return: a numpy array with embedding vectors for all known words
"""
# initialize matrix with zeros
embedding_matrix = np.zeros((len(word_index) + 1, 300))
# loop over all the words
for word, i in word_index.items():
# if word is found in pre-trained embeddings,
# update the matrix. if the word is not found,
# the vector is zeros!
if word in embedding_dict:
embedding_matrix[i] = embedding_dict[word]
# return embedding matrix
return embedding_matrix
def run(df, fold):
"""
Run training and validation for a given fold
and dataset
:param df: pandas dataframe with kfold column
:param fold: current fold, int
"""
# fetch training dataframe
train_df = df[df.kfold != fold].reset_index(drop=True)
# fetch validation dataframe
valid_df = df[df.kfold == fold].reset_index(drop=True)
print("Fitting tokenizer")
# we use tf.keras for tokenization
# you can use your own tokenizer and then you can
# get rid of tensorflow
tokenizer = tf.keras.preprocessing.text.Tokenizer()
tokenizer.fit_on_texts(df.review.values.tolist())
# convert training data to sequences
# for example : "bad movie" gets converted to
# [24, 27] where 24 is the index for bad and 27 is the
# index for movie
xtrain = tokenizer.texts_to_sequences(train_df.review.values)
# similarly convert validation data to
# sequences
xtest = tokenizer.texts_to_sequences(valid_df.review.values)
# zero pad the training sequences given the maximum length
# this padding is done on left hand side
# if sequence is > MAX_LEN, it is truncated on left hand side too
xtrain = tf.keras.preprocessing.sequence.pad_sequences(
xtrain, maxlen=config.MAX_LEN
)
# zero pad the validation sequences
xtest = tf.keras.preprocessing.sequence.pad_sequences(
xtest, maxlen=config.MAX_LEN
)
# initialize dataset class for training
train_dataset = dataset.IMDBDataset(
reviews=xtrain,
targets=train_df.sentiment.values
)
# create torch dataloader for training
# torch dataloader loads the data using dataset
# class in batches specified by batch size
train_data_loader = torch.utils.data.DataLoader(
train_dataset,
batch_size=config.TRAIN_BATCH_SIZE,
num_workers=2
)
# initialize dataset class for validation
valid_dataset = dataset.IMDBDataset(
reviews=xtest,
targets=valid_df.sentiment.values
)
# create torch dataloader for validation
valid_data_loader = torch.utils.data.DataLoader(
valid_dataset,
batch_size=config.VALID_BATCH_SIZE,
num_workers=1
)
print("Loading embeddings")
# load embeddings as shown previously
embedding_dict = load_vectors("../input/crawl-300d-2M.vec")
# this is where we have called the function create_embedding_matrix
embedding_matrix = create_embedding_matrix(
tokenizer.word_index, embedding_dict
)
# create torch device, since we use gpu, we are using cuda
device = torch.device("cuda")
# fetch our LSTM model
model = lstm.LSTM(embedding_matrix)
# send model to device
model.to(device)
# initialize Adam optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print("Training Model")
# set best accuracy to zero
best_accuracy = 0
# set early stopping counter to zero
early_stopping_counter = 0
# train and validate for all epochs
for epoch in range(config.EPOCHS):
# train one epoch
engine.train(train_data_loader, model, optimizer, device)
# validate
outputs, targets = engine.evaluate(
valid_data_loader, model, device
)
# use threshold of 0.5
# please note we are using linear layer and no sigmoid
# you should do this 0.5 threshold after sigmoid
outputs = np.array(outputs) >= 0.5
# calculate accuracy
accuracy = metrics.accuracy_score(targets, outputs)
print(
f"FOLD:{fold}, Epoch: {epoch}, Accuracy Score = {accuracy}"
)
# simple early stopping
if accuracy > best_accuracy:
best_accuracy = accuracy
else:
early_stopping_counter += 1
if early_stopping_counter > 2:
break

if __name__ == "__main__":
# load data
df = pd.read_csv("../input/imdb_folds.csv")
# train for all folds
run(df, fold=0)
run(df, fold=1)
run(df, fold=2)
run(df, fold=3)
run(df, fold=4)

